# Chapter 8: Foundation Models (Chronos + TimesFM) on EUR/RON

Run this notebook on **Google Colab** (GPU recommended but CPU works).

It downloads EUR/RON data, runs Chronos and TimesFM zero-shot forecasts,
generates the two foundation charts, and prints results for the LaTeX slides.

In [ ]:
!pip install -q yfinance chronos-forecasting torch
# TimesFM needs specific Python/JAX — try installing:
!pip install -q timesfm 2>/dev/null || echo 'TimesFM install failed (Python version issue) — will skip'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.metrics import mean_squared_error, mean_absolute_error
import time, warnings, json
warnings.filterwarnings('ignore')

# Style
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.size'] = 8
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['lines.linewidth'] = 0.75

MAIN_BLUE = '#1A3A6E'
IDA_RED   = '#DC3545'
FOREST    = '#2E7D32'
AMBER     = '#E67E22'
PURPLE    = '#7B2D8E'
TEAL      = '#00897B'

In [ ]:
# Download EUR/RON data
eurron_raw = yf.download('EURRON=X', start='2019-01-01', end='2025-06-01', progress=False)
if isinstance(eurron_raw.columns, pd.MultiIndex):
    eurron = eurron_raw['Close']['EURRON=X'].dropna()
else:
    eurron = eurron_raw['Close'].dropna()
eurron = pd.Series(eurron.values.flatten(), index=eurron.index, name='EURRON')

n = len(eurron)
val_end = int(n * 0.85)
price_test = eurron.iloc[val_end:]
test_idx = price_test.index
context_data = eurron.iloc[:val_end].values.astype(np.float32)

print(f'Data: {n} obs, Context: {val_end}, Test: {len(price_test)}')
print(f'Test period: {test_idx[0].strftime("%Y-%m-%d")} to {test_idx[-1].strftime("%Y-%m-%d")}')
print(f'Test range: {price_test.min():.4f} – {price_test.max():.4f}')

In [ ]:
# Classical baselines (quick ARIMA for reference)
from statsmodels.tsa.arima.model import ARIMA

t0 = time.time()
all_prices = eurron.values.astype(float)
arima_preds = np.zeros(len(price_test))
history = list(all_prices[:val_end])
for i in range(len(price_test)):
    fit_i = ARIMA(history, order=(1,1,1)).fit()
    arima_preds[i] = fit_i.forecast(steps=1)[0]
    history.append(all_prices[val_end + i])
arima_time = time.time() - t0
arima_rmse = np.sqrt(mean_squared_error(price_test.values, arima_preds))
arima_mae = mean_absolute_error(price_test.values, arima_preds)
print(f'ARIMA(1,1,1): RMSE={arima_rmse:.4f}, MAE={arima_mae:.4f}, Time={arima_time:.1f}s')

In [ ]:
# ===== CHRONOS =====
import torch
from chronos import ChronosPipeline

t0 = time.time()
pipeline = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-small',
    device_map='cpu',
    torch_dtype=torch.float32,
)

context = torch.tensor(context_data, dtype=torch.float32).unsqueeze(0)
horizon = len(price_test)

forecast = pipeline.predict(context, horizon, num_samples=20)
chronos_median = forecast[0].median(dim=0).values.numpy()
chronos_lo = np.quantile(forecast[0].numpy(), 0.1, axis=0)
chronos_hi = np.quantile(forecast[0].numpy(), 0.9, axis=0)
chronos_time = time.time() - t0

n_chr = min(len(chronos_median), len(price_test))
chronos_rmse = np.sqrt(mean_squared_error(price_test.values[:n_chr], chronos_median[:n_chr]))
chronos_mae = mean_absolute_error(price_test.values[:n_chr], chronos_median[:n_chr])

print(f'Chronos: RMSE={chronos_rmse:.4f}, MAE={chronos_mae:.4f}, Time={chronos_time:.1f}s')
print(f'Chronos range: {chronos_median.min():.4f} – {chronos_median.max():.4f}')

In [ ]:
# ===== TIMESFM =====
HAS_TIMESFM = False
timesfm_rmse = timesfm_mae = timesfm_time = None
timesfm_preds = None

try:
    import timesfm
    t0 = time.time()
    tfm = timesfm.TimesFm(
        context_len=512,
        horizon_len=len(price_test),
        input_patch_len=32,
        output_patch_len=128,
        num_layers=20,
        model_dims=1280,
        backend='cpu',
    )
    tfm.load_from_checkpoint(repo_id='google/timesfm-1.0-200m')
    point_forecast, _ = tfm.forecast([context_data])
    timesfm_preds = point_forecast[0][:len(price_test)]
    timesfm_time = time.time() - t0
    n_tfm = min(len(timesfm_preds), len(price_test))
    timesfm_rmse = np.sqrt(mean_squared_error(price_test.values[:n_tfm], timesfm_preds[:n_tfm]))
    timesfm_mae = mean_absolute_error(price_test.values[:n_tfm], timesfm_preds[:n_tfm])
    HAS_TIMESFM = True
    print(f'TimesFM: RMSE={timesfm_rmse:.4f}, MAE={timesfm_mae:.4f}, Time={timesfm_time:.1f}s')
except Exception as e:
    print(f'TimesFM not available: {e}')

In [ ]:
# ===== CHART 1: Foundation Comparison =====
fm_names  = ['ARIMA(1,1,1)', 'Chronos']
fm_rmse   = [arima_rmse, chronos_rmse]
fm_mae    = [arima_mae, chronos_mae]
fm_colors = [MAIN_BLUE, PURPLE]

if HAS_TIMESFM:
    fm_names.append('TimesFM')
    fm_rmse.append(timesfm_rmse)
    fm_mae.append(timesfm_mae)
    fm_colors.append(TEAL)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
x2 = np.arange(len(fm_names))
w = 0.35

bars_r = axes[0].bar(x2 - w/2, fm_rmse, w, color=fm_colors,
                     alpha=0.9, edgecolor='white', linewidth=0.5, label='RMSE')
bars_m = axes[0].bar(x2 + w/2, fm_mae, w, color=fm_colors,
                     alpha=0.5, edgecolor='white', linewidth=0.5, label='MAE')
axes[0].set_ylabel('Error')
axes[0].set_title('Classical vs Foundation Models', fontweight='bold')
axes[0].set_xticks(x2)
axes[0].set_xticklabels(fm_names, rotation=15, ha='right', fontsize=8)
for bar in bars_r:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.0001,
                 f'{h:.4f}', ha='center', va='bottom', fontsize=7)
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)

# Right: relative improvement
rel = [(arima_rmse - r) / arima_rmse * 100 for r in fm_rmse]
bar_c = [FOREST if v > 0 else IDA_RED for v in rel]
axes[1].bar(x2, rel, color=bar_c, alpha=0.8, edgecolor='white', linewidth=0.5)
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].set_ylabel('RMSE Improvement vs ARIMA (%)')
axes[1].set_title('Relative Performance', fontweight='bold')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(fm_names, rotation=15, ha='right', fontsize=8)
for i, v in enumerate(rel):
    axes[1].text(i, v + (0.3 if v >= 0 else -0.6), f'{v:.1f}%', ha='center', fontsize=7)

plt.tight_layout()
plt.savefig('ch8_foundation_comparison.pdf', bbox_inches='tight', transparent=True, dpi=300)
plt.savefig('ch8_foundation_comparison.png', bbox_inches='tight', transparent=True, dpi=300)
plt.show()
print('Saved: ch8_foundation_comparison.pdf')

In [ ]:
# ===== CHART 2: Foundation Predictions Detail =====
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test_idx, price_test.values, color='#333333', linewidth=1.5,
        label='Actual EUR/RON', zorder=5)
ax.plot(test_idx, arima_preds, color=MAIN_BLUE, linewidth=0.8,
        linestyle='--', alpha=0.6, label='ARIMA (baseline)')

ax.plot(test_idx[:n_chr], chronos_median[:n_chr],
        color=PURPLE, linewidth=1.2, label='Chronos (zero-shot)')
ax.fill_between(test_idx[:n_chr],
                chronos_lo[:n_chr], chronos_hi[:n_chr],
                color=PURPLE, alpha=0.15, label='Chronos 80% CI')

if HAS_TIMESFM:
    ax.plot(test_idx[:n_tfm], timesfm_preds[:n_tfm],
            color=TEAL, linewidth=1.2, label='TimesFM (zero-shot)')

ax.set_xlabel('Date')
ax.set_ylabel('EUR/RON Exchange Rate')
ax.set_title('Foundation Models: Zero-Shot Predictions on EUR/RON', fontweight='bold')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=4, frameon=False)
plt.tight_layout()
plt.savefig('ch8_foundation_predictions.pdf', bbox_inches='tight', transparent=True, dpi=300)
plt.savefig('ch8_foundation_predictions.png', bbox_inches='tight', transparent=True, dpi=300)
plt.show()
print('Saved: ch8_foundation_predictions.pdf')

In [ ]:
# ===== RESULTS SUMMARY =====
print('=' * 60)
print('RESULTS — copy to LaTeX slides')
print('=' * 60)
print(f'ARIMA(1,1,1):        RMSE={arima_rmse:.4f}, MAE={arima_mae:.4f}, Time={arima_time:.1f}s')
print(f'Chronos (zero-shot): RMSE={chronos_rmse:.4f}, MAE={chronos_mae:.4f}, Time={chronos_time:.1f}s')
if HAS_TIMESFM:
    print(f'TimesFM (zero-shot): RMSE={timesfm_rmse:.4f}, MAE={timesfm_mae:.4f}, Time={timesfm_time:.1f}s')

print()
print('LaTeX table rows:')
print(f'    Chronos (zero-shot) & {chronos_rmse:.4f} & {chronos_mae:.4f} & {chronos_time:.1f} & No \\\\')
if HAS_TIMESFM:
    print(f'    TimesFM (zero-shot) & {timesfm_rmse:.4f} & {timesfm_mae:.4f} & {timesfm_time:.1f} & No \\\\')

print()
print('Download ch8_foundation_comparison.pdf and ch8_foundation_predictions.pdf')
print('and place them in the charts/ directory of the TSA repository.')

In [ ]:
# Download files from Colab
try:
    from google.colab import files
    files.download('ch8_foundation_comparison.pdf')
    files.download('ch8_foundation_predictions.pdf')
    files.download('ch8_foundation_comparison.png')
    files.download('ch8_foundation_predictions.png')
except:
    print('Not running on Colab — files saved to current directory')